# Connections Solver


This notebook displays an overview of our project. Each section can be run independently.

### Data example
This section shows an example of the data we use to test our models.

In [28]:

from data_loader import get_train_test_split


ds_train, ds_test = get_train_test_split()
# only small amout of test puzzles for showcase
ds_test = ds_test.select(range(30))
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

test_sample = ds_test.select(range(1))
print("Example test data: " + str(test_sample[0]))

Training puzzles: 521
Testing puzzles: 30
Example test data: {'date': Timestamp('2023-07-07 00:00:00'), 'contest': 'NYT Connections 26 - July 7th, 2023', 'words': ['DENMARK', 'GREECE', 'POLAND', 'PORTUGAL', 'COPY', 'ECHO', 'MIMIC', 'PARROT', 'CRUISE', 'HOLLAND', 'PETTY', 'WAITS', 'DILL', 'LIVID', 'MILD', 'MIX'], 'answers': [{'answerDescription': 'EUROPEAN COUNTRIES', 'words': ['DENMARK', 'GREECE', 'POLAND', 'PORTUGAL']}, {'answerDescription': 'SYNONYMS FOR IMITATE', 'words': ['COPY', 'ECHO', 'MIMIC', 'PARROT']}, {'answerDescription': 'TOMS', 'words': ['CRUISE', 'HOLLAND', 'PETTY', 'WAITS']}, {'answerDescription': 'WORDS SPELLED WITH ROMAN NUMERALS', 'words': ['DILL', 'LIVID', 'MILD', 'MIX']}], 'difficulty': None}


## Models

### DeBERTa fine-tuned

#### Instructions for running:
1. Run each of the cells in this section.
The output of this cell is our pre-ran results of the DeBERTa fine-tuned notebook. To see in-depth implementation run cells in DeBERTa-lora-final.ipynb

In [20]:
import json
from pathlib import Path

results_base = Path("reports")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], reverse=True)
run_dir = next((d for d in run_dirs if (d / "deberta_lora_metrics.json").exists()), None)
if run_dir is None:
    print("No timestamped run found with deberta_lora_metrics.json")
else:
    print(f"Run folder: {run_dir}\n")
    for name, path in [
        ("LoRA (train)", "deberta_lora_metrics.json"),
        ("LoRA (test)", "deberta_lora_test_metrics.json"),
    ]:
        p = run_dir / path
        if p.exists():
            m = json.loads(p.read_text())
            print(f"{name}:")
            for k, v in m.items():
                if isinstance(v, float):
                    print(f"  {k}: {v:.4f}")
                else:
                    print(f"  {k}: {v}")
            print()
        else:
            print(f"{name}: (file not found: {path})\n")

Run folder: reports\deberta_lora

LoRA (train):
  split: train
  zero_one_accuracy: 0.0058
  mean_min_swaps: 3.4184
  mean_n_correct_groups: 0.3666
  mean_correct_word_count: 9.6718
  mean_inference_time_sec: 0.1933
  total_inference_time_sec: 100.6883
  n_eval: 521

LoRA (test):
  split: test
  zero_one_accuracy: 0.0000
  mean_min_swaps: 3.4351
  mean_n_correct_groups: 0.3740
  mean_correct_word_count: 9.6565
  mean_inference_time_sec: 0.1860
  total_inference_time_sec: 24.3687
  n_eval: 131



### LLM Zero and Few-Shot

#### Instructions for running:
    1. Get API KEY from [Cerebras Website]("https://www.cerebras.ai/"). 
    2. Save it to environment as "CEREBRAS_API_KEY".
    3. Run the next two cells (runtime of the second cell is variable)

    Alternatively, just run the last cell in this section to see results quickly.

In [12]:
import os
from cerebras.cloud.sdk import Cerebras
from conn import accuracy_min_swaps, accuracy_zero_one, evaluate
from data_loader import gold_groups_from_row, get_train_test_split
import time
import src.project_notebook_imports_few_shot_LLM as fs

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

ds_train, ds_test = get_train_test_split()
# only small amout of test puzzles for showcase
ds_test = ds_test.select(range(30))
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

Training puzzles: 521
Testing puzzles: 30


In [13]:
N_EVAL = len(ds_test)

results_list = []

few_shot_values = [0, 1, 5]
for k in few_shot_values:

    start = time.time()

    results = evaluate(
        ds_test,
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: fs.solve_puzzle(words, k=k, split_for_few_shot=ds_train, client=client),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=False,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    end = time.time()
    runtime = end - start

    results_list.append({
        "k": k,
        "zero_one_accuracy": acc,
        "mean_swaps": mean_swaps,
        "runtime": runtime,
        "n_eval": n_eval
    })

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")
    print(f"k={k} | Runtime: {runtime:.2f} seconds")


Invalid LLM output: GROUP 1: TEE (GOLF) || TEE (SHIRT) || GEAR || SAW
GROUP 2: TEA || TEE (SHIRT) || ZIPPER || COMB
GROUP 3: TI (MUSICAL NOTE) || GEEZ || NUTS || FUDGE
GROUP 4: RATS || BANK || DELTA || MOUTH
Correct Answer: ['TEA', 'TEE (GOLF)', 'TEE (SHIRT)', 'TI (MUSICAL NOTE)', 'COMB', 'GEAR', 'SAW', 'ZIPPER', 'FUDGE', 'GEEZ', 'NUTS', 'RATS', 'BANK', 'BED', 'DELTA', 'MOUTH']
Retrying (1/4)
Invalid LLM output: GROUP 1: TEE (GOLF) || TEE (SHIRT) || GEAR || SAW
GROUP 2: TEA || TEE (SHIRT) || TEE (GOLF) || ZIPPER
GROUP 3: COMB || GEAR || ZIPPER || SAW
GROUP 4: FUDGE || GEEZ || NUTS || MOUTH
Correct Answer: ['TEA', 'TEE (GOLF)', 'TEE (SHIRT)', 'TI (MUSICAL NOTE)', 'COMB', 'GEAR', 'SAW', 'ZIPPER', 'FUDGE', 'GEEZ', 'NUTS', 'RATS', 'BANK', 'BED', 'DELTA', 'MOUTH']
Retrying (2/4)
Invalid LLM output: GROUP 1: EWE || YEW || THOU || YOU
GROUP 2: BOAT || CREW || SCOOP || BOWL
GROUP 3: U || V || K || 8
GROUP 4: GRAND || GLUE || TUESDAY
Correct Answer: ['EWE', 'U', 'YEW', 'YOU', 'BOAT', 'CREW', 'S

In [22]:
import json
from pathlib import Path

results_base = Path("reports")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], reverse=True)
run_dir = next((d for d in run_dirs if (d / "llm_k0_test_metrics.json").exists()), None)
if run_dir is None:
    print("No timestamped run found with deberta_lora_metrics.json")
else:
    print(f"Run folder: {run_dir}\n")
    for name, path in [
        ("LLM k=0", "llm_k0_test_metrics.json"),
        ("LLM k=1", "llm_k1_test_metrics.json"),
        ("LLM k=3", "llm_k3_test_metrics.json"),
        ("LLM k=5", "llm_k5_test_metrics.json"),
        ("LLM k=10", "llm_k10_test_metrics.json"),
        ("LLM k=20", "llm_k20_test_metrics.json"),
    ]:
        p = run_dir / path
        if p.exists():
            m = json.loads(p.read_text())
            print(f"{name}:")
            for k, v in m.items():
                if isinstance(v, float):
                    print(f"  {k}: {v:.4f}")
                else:
                    print(f"  {k}: {v}")
            print()
        else:
            print(f"{name}: (file not found: {path})\n")

Run folder: reports\LLM

LLM k=0:
  k: 0
  model: llama3.1-8b
  split: test
  zero_one_accuracy: 0.4656
  mean_min_swaps: 1.4504
  mean_n_correct_groups: 2.3206
  mean_correct_word_count: 13.3511
  mean_inference_time_sec: 2.0513
  total_inference_time_sec: 268.7230
  n_eval: 131

LLM k=1:
  k: 1
  model: llama3.1-8b
  split: test
  zero_one_accuracy: 0.7023
  mean_min_swaps: 0.7099
  mean_n_correct_groups: 3.1069
  mean_correct_word_count: 14.6794
  mean_inference_time_sec: 2.8480
  total_inference_time_sec: 373.0844
  n_eval: 131

LLM k=3:
  k: 3
  model: llama3.1-8b
  split: test
  zero_one_accuracy: 0.8308
  mean_min_swaps: 0.3462
  mean_n_correct_groups: 3.5385
  mean_correct_word_count: 15.3462
  mean_inference_time_sec: 2.3315
  total_inference_time_sec: 303.0908
  n_eval: 130

LLM k=5:
  k: 5
  model: llama3.1-8b
  split: test
  zero_one_accuracy: 0.8846
  mean_min_swaps: 0.2154
  mean_n_correct_groups: 3.7000
  mean_correct_word_count: 15.5846
  mean_inference_time_sec: 2.1374